In [0]:
# Celda 1: variables de la tabla que NO están en el diccionario
import pandas as pd
from pyspark.sql import functions as F

dic = pd.read_csv("/Volumes/workspace/default/raw/LCDataDictionary.csv")
dic_vars = set(dic["LoanStatNew"].dropna().str.strip())   # strip: hay nombres con espacios al final

cols_raw = spark.table("workspace.default.lc_accepted_raw").columns
fuera = sorted(set(cols_raw) - dic_vars)

print(f"Variables en la tabla: {len(cols_raw)} | en el diccionario: {len(dic_vars)} | sin documentar: {len(fuera)}")
for c in fuera:
    print(c)


In [0]:
# Celda 2: % de faltantes por variable y cosecha (sobre la población final)
pob = spark.table("workspace.default.lc_population")
cols = [c for c in pob.columns if c != "issue_year"]

# Una sola pasada sobre los datos: para cada variable, el promedio de "es nulo" = % de faltantes
faltantes = (pob.groupBy("issue_year")
                .agg(*[F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 1).alias(c) for c in cols]))

# Transponer: filas = variables, columnas = cosechas (como un PROC TRANSPOSE)
pdf = faltantes.toPandas().set_index("issue_year").sort_index().T
pdf.columns = [str(c) for c in pdf.columns]
pdf["max_dev"] = pdf[["2013", "2014"]].max(axis=1)   # peor cosecha de desarrollo
pdf.index.name = "variable"

display(pdf.reset_index().sort_values("max_dev", ascending=False))


In [0]:
from pyspark.sql import functions as F

pob = spark.table("workspace.default.lc_population").filter(F.col("issue_year") >= 2013)

# ¿Los faltantes de utilización coinciden con clientes sin tarjetas bancarias?
display(pob.groupBy(F.col("bc_util").isNull().alias("bc_util_nulo"),
                    (F.col("num_bc_tl") == 0).alias("sin_tarjetas_bc"))
           .agg(F.count("*").alias("casos"),
                F.round(F.avg("target") * 100, 2).alias("bad_rate_pct")))


In [0]:
import pandas as pd
from pyspark.sql import functions as F

pob = spark.table("workspace.default.lc_population")
dev = pob.filter(F.col("sample") == "DEV")

# Listas de exclusión por motivo (decisiones de las sesiones anteriores)
CONTROL = ["id", "issue_d", "issue_date", "issue_year", "term", "term_months",
           "loan_status", "gbi", "target", "sample"]
FUGA = ["total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
        "recoveries", "collection_recovery_fee", "out_prncp", "out_prncp_inv",
        "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
        "last_fico_range_high", "last_fico_range_low", "pymnt_plan",
        "debt_settlement_flag", "debt_settlement_flag_date", "deferral_term",
        "payment_plan_start_date", "orig_projected_additional_accrued_interest"]
FUGA_PREFIJOS = ("hardship_", "settlement_")
SALIDA_LC = ["grade", "sub_grade", "int_rate", "installment"]   # installment se calcula con int_rate
ID_TEXTO = ["member_id", "url", "desc", "title", "emp_title", "zip_code"]

# Una sola pasada sobre DEV: % de faltantes y número de valores distintos por variable
cols = pob.columns
stats = dev.agg(*[F.avg(F.col(c).isNull().cast("int")).alias(f"{c}__nul") for c in cols],
                *[F.approx_count_distinct(c).alias(f"{c}__dist") for c in cols]).first().asDict()

def motivo(c):
    if c in CONTROL:                                 return "Control (no es predictor)"
    if c in FUGA or c.startswith(FUGA_PREFIJOS):     return "Fuga: posterior a la originación"
    if c in SALIDA_LC:                               return "Salida del modelo LC (benchmark)"
    if c in ID_TEXTO:                                return "Identificador o texto libre"
    if stats[f"{c}__nul"] >= 0.95:                   return "Vacía en DEV (>=95% faltantes)"
    if stats[f"{c}__dist"] <= 1:                     return "Constante en DEV"
    return "Candidata"

inv = pd.DataFrame({"variable": cols,
                    "pct_faltantes_dev": [round(stats[f"{c}__nul"] * 100, 1) for c in cols],
                    "valores_distintos_dev": [stats[f"{c}__dist"] for c in cols]})
inv["motivo"] = inv["variable"].apply(motivo)
inv["decision"] = inv["motivo"].apply(lambda m: "Candidata" if m == "Candidata" else "Excluida")

# Se guarda como tabla: es el documento de exclusiones para la validación
spark.createDataFrame(inv).write.mode("overwrite").saveAsTable("workspace.default.lc_variable_inventory")

display(inv.groupby("motivo").size().reset_index(name="variables"))
print(sorted(inv.loc[inv["decision"] == "Candidata", "variable"]))
